In [ ]:
import numpy as np
import pandas as pd
import joblib

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score)

In [ ]:
data = load_breast_cancer()
X = data.data
y = data.target

print(X.shape)
print(np.bincount(y))

In [ ]:

def split_train_val_test(X, y, test_size=0.2, val_size=0.2, random_state=42):

    if val_size <= 0:
        raise ValueError("val_size doit être > 0")

    X_temp, X_test, y_temp, y_test = train_test_split(
        X, y,
        test_size=test_size,
        random_state=random_state,
        stratify=y
    )

    val_ratio = val_size / (1 - test_size)

    X_train, X_val, y_train, y_val = train_test_split(
        X_temp,
        y_temp,
        test_size=val_ratio,
        random_state=random_state,
        stratify=y_temp
    )

    return X_train, X_val, X_test, y_train, y_val, y_test

X_train, X_val, X_test, y_train, y_val, y_test = split_train_val_test(X,y)

print(len(X_train), len(X_val), len(X_test))


In [ ]:

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)


In [ ]:

def bootstrap_scores(model, X, y, n_iterations=30):

    rng = np.random.default_rng(42)
    scores = []

    for _ in range(n_iterations):

        idx = rng.choice(len(X), len(X), replace=True)
        oob = np.setdiff1d(np.arange(len(X)), idx)

        if len(oob) == 0:
            continue

        X_boot = X[idx]
        y_boot = y[idx]

        model.fit(X_boot, y_boot)

        score = model.score(X[oob], y[oob])
        scores.append(score)

    print("Moyenne :", np.mean(scores))
    print("Std :", np.std(scores))

    return scores

rf = RandomForestClassifier(random_state=42)

bootstrap_scores(rf, X_train_scaled, y_train)


In [ ]:

def evaluer_en_cross_val(model, X, y, k=5):

    cv = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)

    scores = cross_val_score(model, X, y, cv=cv, scoring="accuracy")

    print(scores)
    print("Moyenne =", scores.mean())
    print("Std =", scores.std())

rf = RandomForestClassifier(random_state=42)

evaluer_en_cross_val(rf, X_train_scaled, y_train)


In [ ]:

def rapport_metier(y_true, y_pred, cout_fn=10, cout_fp=1):

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    cout_total = fn * cout_fn + fp * cout_fp

    print("Precision:", precision)
    print("Recall:", recall)
    print("F1:", f1)
    print("Coût métier:", cout_total)

rf = RandomForestClassifier(random_state=42)
rf.fit(X_train_scaled, y_train)

pred = rf.predict(X_test_scaled)

rapport_metier(y_test, pred)


In [ ]:
joblib.dump(
    {
        'modele': rf,
        'scaler': scaler
    },
    'modele.joblib'
)

print("Sauvegardé")

In [ ]:

api_code = '''
import joblib
from flask import Flask, request, jsonify

app = Flask(__name__)

bundle = joblib.load("modele.joblib")

model = bundle["modele"]
scaler = bundle["scaler"]

@app.route("/predict", methods=["POST"])
def predict():

    data = request.get_json()

    if "features" not in data:
        return jsonify({"error":"features manquant"}),400

    features = data["features"]

    X = scaler.transform([features])

    pred = int(model.predict(X)[0])
    proba = float(model.predict_proba(X)[0][1])

    return jsonify({
        "prediction": pred,
        "proba": proba
    })

app.run()
'''
print(api_code)


In [ ]:

streamlit_code = '''
import streamlit as st
import joblib

bundle = joblib.load("modele.joblib")

model = bundle["modele"]
scaler = bundle["scaler"]

st.title("Cancer Predictor")

features = []

for i in range(30):
    features.append(st.number_input(f"Feature {i+1}", value=0.0))

if st.button("Prédire"):
    X = scaler.transform([features])
    pred = model.predict(X)[0]
    proba = model.predict_proba(X)[0][1]

    st.write(pred)
    st.write(proba)
'''
print(streamlit_code)


In [ ]:

rf = RandomForestClassifier(random_state=42)

mlp = MLPClassifier(
    hidden_layer_sizes=(16,8),
    max_iter=500,
    random_state=42
)

rf_scores = cross_val_score(rf, X_train_scaled, y_train, cv=5)

mlp_scores = cross_val_score(mlp, X_train_scaled, y_train, cv=5)

print("RF :", rf_scores.mean())
print("MLP :", mlp_scores.mean())
